# Government Scheme Dataset Creation

This notebook contains the complete implementation for creating, validating, and saving a high-quality dataset of Indian Government Schemes for youth and employment.

### Scheme Categories Covered:
* **Skill Development**
* **Internships**
* **Apprenticeships**
* **Scholarships**
* **Employment Assistance**

In [1]:
# Install required libraries for this notebook
%pip install pandas numpy matplotlib seaborn scikit-learn scipy requests tqdm openpyxl pydantic

Note: you may need to restart the kernel to use updated packages.


## 1. Import Libraries

First, we import the necessary Python libraries for data structures, serialization, data manipulation (Pandas), and validation (Pydantic).

In [2]:
import json
from typing import List, Optional, Union
import pandas as pd
from pydantic import BaseModel, Field, ValidationError, ConfigDict

print("Libraries imported successfully!")

Libraries imported successfully!


## 2. Define the Pydantic Schema

We define the `Scheme` Pydantic model. This model enforces structure, validation constraints (e.g., non-negative age limits), and types for all 16 mandatory fields of each government scheme.

In [3]:
class Scheme(BaseModel):
    model_config = ConfigDict(populate_by_name=True)

    scheme_id: str = Field(..., description="Unique identifier for the scheme")
    scheme_name: str = Field(..., description="Official name of the scheme")
    category: str = Field(..., description="Category: Skill Development, Internships, Apprenticeships, Scholarships, or Employment Assistance")
    ministry: str = Field(..., description="Indian government ministry overseeing the scheme")
    description: str = Field(..., description="Detailed description of the scheme's purpose and activities")
    min_age: Optional[int] = Field(None, ge=0, description="Minimum age eligibility in years")
    max_age: Optional[int] = Field(None, ge=0, description="Maximum age eligibility in years")
    gender: Union[str, List[str]] = Field(..., description="Eligible gender(s): e.g., All, Female, etc.")
    occupation: Union[str, List[str]] = Field(..., description="Target occupation(s): e.g., Students, Unemployed, etc.")
    education: Union[str, List[str]] = Field(..., description="Minimum educational qualifications required")
    income_limit: Optional[Union[int, float, str]] = Field(None, description="Maximum family income limit if applicable")
    state: str = Field("All India", description="State applicability: 'All India' or specific state name")
    benefits: Union[str, List[str]] = Field(..., description="Benefits provided under the scheme")
    required_documents: List[str] = Field(..., description="Documents needed to apply")
    application_link: str = Field(..., description="Direct link to application form/portal")
    official_scheme_url: str = Field(..., description="Official webpage URL for the scheme")

print("Pydantic model 'Scheme' successfully defined!")

Pydantic model 'Scheme' successfully defined!


## 3. Load the Government Schemes Dataset

We load the raw dataset of 20 real government schemes from the `raw_schemes.json` file.

In [4]:
# Load the raw dataset from JSON file
with open("raw_schemes.json", "r", encoding="utf-8") as f:
    raw_schemes = json.load(f)

print(f"Successfully loaded {len(raw_schemes)} raw schemes from 'raw_schemes.json'!")

Successfully loaded 20 raw schemes from 'raw_schemes.json'!


## 4. Run Pydantic Validation

We loop through the raw list of dictionaries, validate each dictionary against the Pydantic `Scheme` model to ensure complete compliance, and collect the verified schemas.

In [5]:
validated_schemes = []
validation_errors = []

for scheme_dict in raw_schemes:
    try:
        # Validate and parse using Pydantic model
        validated_scheme = Scheme(**scheme_dict)
        validated_schemes.append(validated_scheme.model_dump())
    except ValidationError as e:
        print(f"Validation failed for: {scheme_dict.get('scheme_name')}")
        validation_errors.append((scheme_dict, e))

if not validation_errors:
    print(f"Success! All {len(validated_schemes)} schemes validated successfully with no validation errors.")
else:
    print(f"Validation failed for {len(validation_errors)} records. Details:")
    for record, err in validation_errors:
        print(f"Record ID: {record.get('scheme_id')} | Error: {err}")

Success! All 20 schemes validated successfully with no validation errors.


## 5. Convert to Pandas DataFrame and Save to JSON

Next, we load the validated python list of dictionaries into a Pandas DataFrame and export the dataset as a structured `schemes.json` file.

In [6]:
# Convert to DataFrame
df = pd.DataFrame(validated_schemes)

# Export to JSON
json_file_name = "schemes.json"
with open(json_file_name, "w", encoding="utf-8") as f:
    json.dump(validated_schemes, f, indent=4, ensure_ascii=False)

print(f"Dataset successfully saved as '{json_file_name}' and initialized as a Pandas DataFrame!")

Dataset successfully saved as 'schemes.json' and initialized as a Pandas DataFrame!


## 6. Dataset Summaries & Missing Value Audits

Finally, we run a profile of our dataset to show the sizes, preview records, and check for missing values.

In [7]:
print("=== Dataset Size ===")
print(f"Total Schemes: {len(df)}")
print(f"Columns: {list(df.columns)}")
print(f"Distribution of Categories:\n{df['category'].value_counts()}\n")

print("=== Missing Values Check ===")
print(df.isnull().sum())
print("\n")

print("=== Sample Records Preview ===")
# Select a subset of columns for displaying a clean tabular preview
preview_columns = ['scheme_id', 'scheme_name', 'category', 'ministry', 'min_age', 'max_age', 'state']
print(df[preview_columns].head(5).to_string(index=False))

=== Dataset Size ===
Total Schemes: 20
Columns: ['scheme_id', 'scheme_name', 'category', 'ministry', 'description', 'min_age', 'max_age', 'gender', 'occupation', 'education', 'income_limit', 'state', 'benefits', 'required_documents', 'application_link', 'official_scheme_url']
Distribution of Categories:
category
Skill Development        7
Scholarships             5
Employment Assistance    4
Apprenticeships          2
Internships              2
Name: count, dtype: int64

=== Missing Values Check ===
scheme_id              0
scheme_name            0
category               0
ministry               0
description            0
min_age                0
max_age                1
gender                 0
occupation             0
education              0
income_limit           9
state                  0
benefits               0
required_documents     0
application_link       0
official_scheme_url    0
dtype: int64


=== Sample Records Preview ===
scheme_id                                        